Khởi tạo database

In [1]:
import sqlite3
import pandas as pd

# Tạo / mở database
conn = sqlite3.connect("bai_tap_chuong_2.db")
cursor = conn.cursor()

# Xóa bảng cũ nếu có
cursor.execute("DROP TABLE IF EXISTS student")
cursor.execute("DROP TABLE IF EXISTS course")

# Tạo bảng course
cursor.execute("""
CREATE TABLE course (
    id INTEGER PRIMARY KEY,
    course_name TEXT
)
""")

# Tạo bảng student
cursor.execute("""
CREATE TABLE student (
    student_id INTEGER PRIMARY KEY,
    name TEXT,
    class TEXT,
    course_id INTEGER,
    score REAL
)
""")

# Thêm dữ liệu vào bảng course
course_data = [
    (12, 'Giai tich'),
    (34, 'Thong ke'),
    (26, 'Tin hoc')
]

cursor.executemany("""
INSERT INTO course (id, course_name)
VALUES (?, ?)
""", course_data)

# Thêm dữ liệu vào bảng student
student_data = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]

cursor.executemany("""
INSERT INTO student (student_id, name, class, course_id, score)
VALUES (?, ?, ?, ?, ?)
""", student_data)

conn.commit()

print("Đã tạo xong dữ liệu ban đầu.")

Đã tạo xong dữ liệu ban đầu.


# Câu 1: Kết nối 2 bảng

Sử dụng tích Decartes

In [2]:
sql_cartesian = """
SELECT s.student_id, s.name, s.class, s.course_id, s.score,
       c.id AS course_table_id, c.course_name
FROM student s
CROSS JOIN course c
ORDER BY s.student_id, c.id
"""
df_cartesian = pd.read_sql(sql_cartesian, conn)
print("Số dòng của tích Descartes:", len(df_cartesian))
display(df_cartesian)

Số dòng của tích Descartes: 30


,student_id,name,class,course_id,score,course_table_id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


Sử dụng JOIN

In [3]:
# INNER JOIN
sql_inner = """
SELECT s.student_id, s.name, s.class, s.course_id, s.score,
       c.course_name
FROM student s
INNER JOIN course c
    ON s.course_id = c.id
ORDER BY s.student_id
"""
df_inner = pd.read_sql(sql_inner, conn)
display(df_inner)

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke


In [4]:
# LEFT JOIN
sql_left = """
SELECT s.student_id, s.name, s.class, s.course_id, s.score,
       c.course_name
FROM student s
LEFT JOIN course c
    ON s.course_id = c.id
ORDER BY s.student_id
"""
df_left = pd.read_sql(sql_left, conn)
display(df_left)

,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


In [5]:
# RIGHT JOIN
sql_right = """
SELECT s.student_id, s.name, s.class, s.course_id, s.score,
       c.id AS course_id_from_course, c.course_name
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
ORDER BY c.id, s.student_id
"""
df_right = pd.read_sql(sql_right, conn)
display(df_right)

,student_id,name,class,course_id,score,course_id_from_course,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,NaN,None,None,NaN,NaN,26,Tin hoc
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
3,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34,Thong ke


In [6]:
# FULL OUTER JOIN
sql_full = """
SELECT s.student_id, s.name, s.class, s.course_id, s.score,
       c.id AS course_table_id, c.course_name
FROM student s
LEFT JOIN course c
    ON s.course_id = c.id

UNION

SELECT s.student_id, s.name, s.class, s.course_id, s.score,
       c.id AS course_table_id, c.course_name
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
ORDER BY student_id
"""
df_full = pd.read_sql(sql_full, conn)
display(df_full)

,student_id,name,class,course_id,score,course_table_id,course_name
0,NaN,None,None,NaN,NaN,26.0,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12.0,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34.0,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,NaN,None
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,NaN,None
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,NaN,None
6,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,NaN,None
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34.0,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,NaN,None
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,NaN,None


# Câu 2: 

In [7]:
# Dữ liệu trước khi làm sạch
df_before = pd.read_sql("SELECT * FROM student ORDER BY student_id", conn)
display(df_before)

,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0


In [8]:
# Cập nhật course_id còn thiếu
cursor.execute("""
UPDATE student
SET course_id = 26
WHERE course_id IS NULL
""")
conn.commit()

In [ ]:
# Loại bỏ các bản ghi có course_id không tồn tại trong bảng course
cursor.execute("""
DELETE FROM student
WHERE course_id NOT IN (SELECT id FROM course)
""")
conn.commit()

In [10]:
# Dữ liệu sau khi làm sạch
df_clean = pd.read_sql("SELECT * FROM student ORDER BY student_id", conn)
display(df_clean)

,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12,6.7
1,2,Tran Thi Lan,Kinh Te,34,9.2
2,3,Pham Van Nam,Toan Tin,26,7.9
3,7,Bui Tien Dung,Kinh Te,34,9.2
4,9,Duong Huu Phuc,Kinh Te,26,7.2
5,10,Cao Thi Hanh,May Tinh,26,7.0


In [11]:
# 2a
sql_2a = """
SELECT class,
       COUNT(*) AS tong_so_sinh_vien,
       ROUND(AVG(score), 2) AS diem_trung_binh
FROM student
GROUP BY class
ORDER BY class
"""
df_2a = pd.read_sql(sql_2a, conn)
display(df_2a)

,class,tong_so_sinh_vien,diem_trung_binh
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


In [12]:
# 2b 
sql_2b = """
SELECT c.id AS course_id,
       c.course_name,
       COUNT(s.student_id) AS tong_so_sinh_vien,
       ROUND(AVG(s.score), 2) AS diem_trung_binh
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
GROUP BY c.id, c.course_name
ORDER BY c.id
"""
df_2b = pd.read_sql(sql_2b, conn)
display(df_2b)

,course_id,course_name,tong_so_sinh_vien,diem_trung_binh
0,12,Giai tich,1,6.70
1,26,Tin hoc,3,7.37
2,34,Thong ke,2,9.20


In [13]:
# 2c 
sql_2c = """
SELECT c.id AS course_id,
       c.course_name,
       ROUND(AVG(s.score), 2) AS diem_trung_binh,
       CASE
           WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
           WHEN AVG(s.score) >= 6.0 AND AVG(s.score) <= 8.9 THEN 'Tot'
           ELSE 'Kem'
       END AS xep_loai_thi_dua
FROM course c
LEFT JOIN student s
    ON s.course_id = c.id
GROUP BY c.id, c.course_name
ORDER BY c.id
"""
df_2c = pd.read_sql(sql_2c, conn)
display(df_2c)

,course_id,course_name,diem_trung_binh,xep_loai_thi_dua
0,12,Giai tich,6.70,Tot
1,26,Tin hoc,7.37,Tot
2,34,Thong ke,9.20,Xuat sac


# Câu 3: Sắp xếp hạng sinh viên

In [14]:
# 3a
sql_rank_all = """
SELECT student_id, name, class, course_id, score,
       RANK() OVER (ORDER BY score DESC) AS rank_desc,
       RANK() OVER (ORDER BY score ASC) AS rank_asc
FROM student
ORDER BY score DESC, student_id
"""
df_rank_all = pd.read_sql(sql_rank_all, conn)
display(df_rank_all)

,student_id,name,class,course_id,score,rank_desc,rank_asc
0,2,Tran Thi Lan,Kinh Te,34,9.2,1,5
1,7,Bui Tien Dung,Kinh Te,34,9.2,1,5
2,3,Pham Van Nam,Toan Tin,26,7.9,3,4
3,9,Duong Huu Phuc,Kinh Te,26,7.2,4,3
4,10,Cao Thi Hanh,May Tinh,26,7.0,5,2
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,6,1


In [15]:
# Top 3 cao nhất toàn bộ
sql_top3_all = """
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (ORDER BY score DESC, student_id ASC) AS rn
    FROM student
)
WHERE rn <= 3
ORDER BY rn
"""
df_top3_all = pd.read_sql(sql_top3_all, conn)
display(df_top3_all)

,student_id,name,class,course_id,score,rn
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,2
2,3,Pham Van Nam,Toan Tin,26,7.9,3


In [16]:
# Top 3 thấp nhất toàn bộ
sql_bottom3_all = """
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (ORDER BY score ASC, student_id ASC) AS rn
    FROM student
)
WHERE rn <= 3
ORDER BY rn
"""
df_bottom3_all = pd.read_sql(sql_bottom3_all, conn)
display(df_bottom3_all)

,student_id,name,class,course_id,score,rn
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,2
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3


In [17]:
# 3b, Xếp hạng theo điểm số trong từng lớp
sql_rank_class = """
SELECT student_id, name, class, course_id, score,
       RANK() OVER (PARTITION BY class ORDER BY score DESC) AS rank_in_class_desc,
       RANK() OVER (PARTITION BY class ORDER BY score ASC) AS rank_in_class_asc
FROM student
ORDER BY class, score DESC, student_id
"""
df_rank_class = pd.read_sql(sql_rank_class, conn)
display(df_rank_class)

,student_id,name,class,course_id,score,rank_in_class_desc,rank_in_class_asc
0,2,Tran Thi Lan,Kinh Te,34,9.2,1,2
1,7,Bui Tien Dung,Kinh Te,34,9.2,1,2
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3,1
3,10,Cao Thi Hanh,May Tinh,26,7.0,1,2
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2,1
5,3,Pham Van Nam,Toan Tin,26,7.9,1,1


In [18]:
# Top 3 cao nhất theo từng lớp
sql_top3_class = """
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (PARTITION BY class ORDER BY score DESC, student_id ASC) AS rn
    FROM student
)
WHERE rn <= 3
ORDER BY class, rn
"""
df_top3_class = pd.read_sql(sql_top3_class, conn)
display(df_top3_class)

,student_id,name,class,course_id,score,rn
0,2,Tran Thi Lan,Kinh Te,34,9.2,1
1,7,Bui Tien Dung,Kinh Te,34,9.2,2
2,9,Duong Huu Phuc,Kinh Te,26,7.2,3
3,10,Cao Thi Hanh,May Tinh,26,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,12,6.7,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


In [19]:
# Top 3 thấp nhất theo từng lớp
sql_bottom3_class = """
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (PARTITION BY class ORDER BY score ASC, student_id ASC) AS rn
    FROM student
)
WHERE rn <= 3
ORDER BY class, rn
"""
df_bottom3_class = pd.read_sql(sql_bottom3_class, conn)
display(df_bottom3_class)

,student_id,name,class,course_id,score,rn
0,9,Duong Huu Phuc,Kinh Te,26,7.2,1
1,2,Tran Thi Lan,Kinh Te,34,9.2,2
2,7,Bui Tien Dung,Kinh Te,34,9.2,3
3,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
4,10,Cao Thi Hanh,May Tinh,26,7.0,2
5,3,Pham Van Nam,Toan Tin,26,7.9,1


In [20]:
# 3c. xếp hạng theo điểm số trong từng mã môn học
sql_rank_course = """
SELECT student_id, name, class, course_id, score,
       RANK() OVER (PARTITION BY course_id ORDER BY score DESC) AS rank_in_course_desc,
       RANK() OVER (PARTITION BY course_id ORDER BY score ASC) AS rank_in_course_asc
FROM student
ORDER BY course_id, score DESC, student_id
"""
df_rank_course = pd.read_sql(sql_rank_course, conn)
display(df_rank_course)

,student_id,name,class,course_id,score,rank_in_course_desc,rank_in_course_asc
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1,1
1,3,Pham Van Nam,Toan Tin,26,7.9,1,3
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2,2
3,10,Cao Thi Hanh,May Tinh,26,7.0,3,1
4,2,Tran Thi Lan,Kinh Te,34,9.2,1,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,1,1


In [21]:
# Top 3 cao nhất theo từng môn 
sql_top3_course = """
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (PARTITION BY course_id ORDER BY score DESC, student_id ASC) AS rn
    FROM student
)
WHERE rn <= 3
ORDER BY course_id, rn
"""
df_top3_course = pd.read_sql(sql_top3_course, conn)
display(df_top3_course)

,student_id,name,class,course_id,score,rn
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,3,Pham Van Nam,Toan Tin,26,7.9,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
3,10,Cao Thi Hanh,May Tinh,26,7.0,3
4,2,Tran Thi Lan,Kinh Te,34,9.2,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,2


In [22]:
# Top 3 thấp nhất từng môn
sql_bottom3_course = """
SELECT *
FROM (
    SELECT student_id, name, class, course_id, score,
           ROW_NUMBER() OVER (PARTITION BY course_id ORDER BY score ASC, student_id ASC) AS rn
    FROM student
)
WHERE rn <= 3
ORDER BY course_id, rn
"""
df_bottom3_course = pd.read_sql(sql_bottom3_course, conn)
display(df_bottom3_course)

,student_id,name,class,course_id,score,rn
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,1
1,10,Cao Thi Hanh,May Tinh,26,7.0,1
2,9,Duong Huu Phuc,Kinh Te,26,7.2,2
3,3,Pham Van Nam,Toan Tin,26,7.9,3
4,2,Tran Thi Lan,Kinh Te,34,9.2,1
5,7,Bui Tien Dung,Kinh Te,34,9.2,2


# Câu 4: 

In [23]:
# Bổ xung thêm trường graduate_date
cursor.execute("ALTER TABLE student ADD COLUMN graduation_date DATETIME")
conn.commit()

In [24]:
# Cập nhật ngày tốt nghiệp theo hạng điểm
sql_update_grad = """
WITH ranked AS (
    SELECT student_id,
           RANK() OVER (ORDER BY score DESC) AS rnk
    FROM student
)
UPDATE student
SET graduation_date = DATETIME('now', '+' || (
    SELECT rnk
    FROM ranked
    WHERE ranked.student_id = student.student_id
) || ' days')
"""
cursor.execute(sql_update_grad)
conn.commit()

In [25]:
sql_grad = """
SELECT student_id, name, class, course_id, score, graduation_date
FROM student
ORDER BY score DESC, student_id
"""
df_grad = pd.read_sql(sql_grad, conn)
display(df_grad)

,student_id,name,class,course_id,score,graduation_date
0,2,Tran Thi Lan,Kinh Te,34,9.2,2026-04-04 02:56:10
1,7,Bui Tien Dung,Kinh Te,34,9.2,2026-04-04 02:56:10
2,3,Pham Van Nam,Toan Tin,26,7.9,2026-04-06 02:56:10
3,9,Duong Huu Phuc,Kinh Te,26,7.2,2026-04-07 02:56:10
4,10,Cao Thi Hanh,May Tinh,26,7.0,2026-04-08 02:56:10
5,1,Nguyen Minh Hoang,May Tinh,12,6.7,2026-04-09 02:56:10


Kết luận

Sau khi thực hiện kết nối hai bảng student và course, có thể thấy:

Tích Descartes tạo ra toàn bộ tổ hợp giữa 2 bảng, tổng cộng 30 dòng
INNER JOIN chỉ giữ các bản ghi có course_id khớp với id của bảng course
LEFT JOIN giữ toàn bộ sinh viên, kể cả những sinh viên chưa có hoặc có mã môn học không hợp lệ
RIGHT JOIN và FULL OUTER JOIN được mô phỏng trong SQLite do SQLite không hỗ trợ trực tiếp trong nhiều phiên bản

Sau khi làm sạch dữ liệu:

Cập nhật các course_id còn thiếu thành 26 theo giả định hợp lệ
Xóa các bản ghi có course_id không tồn tại trong bảng course

Dữ liệu cuối cùng còn 6 sinh viên hợp lệ.

Thống kê theo lớp
Kinh Te: 3 sinh viên, điểm trung bình 8.53
May Tinh: 2 sinh viên, điểm trung bình 6.85
Toan Tin: 1 sinh viên, điểm trung bình 7.90
Thống kê theo môn học
Giai tich: 1 sinh viên, điểm trung bình 6.70
Tin hoc: 3 sinh viên, điểm trung bình 7.37
Thong ke: 2 sinh viên, điểm trung bình 9.20
Phân loại thi đua
Giai tich: Tốt
Tin hoc: Tốt
Thong ke: Xuất sắc
Xếp hạng toàn bộ

Top 3 cao nhất:

Tran Thi Lan — 9.2
Bui Tien Dung — 9.2
Pham Van Nam — 7.9

Top 3 thấp nhất:

Nguyen Minh Hoang — 6.7
Cao Thi Hanh — 7.0
Duong Huu Phuc — 7.2

Đã bổ sung trường graduation_date kiểu DATETIME và cập nhật theo quy tắc lấy thời gian hiện tại cộng với thứ hạng của sinh viên theo điểm số. Cho thấy sinh viên có điểm càng cao thì tốt nghiệp càng sớm